In [1]:
import argparse
import os
import re
import time
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import cv2
from termcolor import colored
from torch.utils.data import Dataset
import matplotlib.pyplot as plt

from IPython.display import clear_output

from nuscenes import NuScenes

class DarkScenesParsingDataset(Dataset):
    def __init__(self, version, dataroot):
        """
        Initialize the NuScenes dataset for use with PyTorch.
        Serves EGOCENTRIC (Local) coordinates with metadata for Global reconstruction.
        """
        print(colored(f"Initializing NuScenes dataset with version {version} at {dataroot}", "cyan"))
        self.nusc = NuScenes(version=version, dataroot=dataroot, verbose=False)

        # Create list of samples
        self.samples = []
        num_hist_points = 10 # including current
        num_future_points = 10
        for scene in self.nusc.scene:
            nbr_samples = scene['nbr_samples']
            if nbr_samples < num_hist_points + num_future_points: 
                continue
            
            sample = self.nusc.get('sample', scene['first_sample_token'])
            
            # Skip first 9 samples to ensure 10 frames of history
            for _ in range(num_hist_points - 1):
                sample = self.nusc.get('sample', sample['next'])
                
            # Add samples (stop 10 frames before end for future ground truth)
            for _ in range(nbr_samples - 19):
                self.samples.append(sample)
                sample = self.nusc.get('sample', sample['next'])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # 1. Get Current Ego Pose (Reference Frame)
        cam_token = sample['data']['CAM_FRONT']
        cam_data = self.nusc.get('sample_data', cam_token)

        # 5. Load Image
        image_path = os.path.join(self.nusc.dataroot, cam_data['filename'])
#         image = Image.open(image_path).convert('RGB')
        
        image = cv2.imread(image_path)  # BGR format
        
        return {
            'image': image,
            'sample_token': sample['token']
        }
    

# def parse_args():
#     parser = argparse.ArgumentParser(description="Evaluate a single NuScenes sample")
#     parser.add_argument('--dataroot', type=str, default='/storage/ice-shared/cs8803vlm/rmineyev3')
#     parser.add_argument('--version', type=str, default='v1.0-test', help='NuScenes version')
    
#     return parser.parse_args()

def is_dark(img, threshold=60):
    if img is None:
        raise ValueError("Could not read image")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean_brightness = np.mean(gray)
    
    return mean_brightness < threshold, mean_brightness

In [34]:
version = 'v1.0-test'
dataroot = '/storage/ice-shared/cs8803vlm/rmineyev3'

dataset = DarkScenesParsingDataset(
    version=version,
    dataroot=dataroot
)

import time

start_idx = 0
num_samples = len(dataset) - start_idx

num_under_40 = 0
num_under_60 = 0
num_under_80 = 0

output_matrix = np.zeros((len(dataset), 3), dtype=bool) # number of samples vs. (idx + num_bool)

for idx in range(start_idx, len(dataset)):
    sample = dataset[idx]
    image = sample['image']

    # Check if it's a dark image
    img_is_dark, mean_brightness = is_dark(image)
    
    is_under_40 = False
    is_under_60 = False
    is_under_80 = False
    
    if mean_brightness < 40:
        num_under_40 += 1
        is_under_40 = True
    if mean_brightness < 60:
        num_under_60 += 1
        is_under_60 = True
    if mean_brightness < 80:
        num_under_80 += 1
        is_under_80 = True
        
    output_matrix[idx, :] = [is_under_40, is_under_60, is_under_80]
    
#     clear_output(wait=True)
    
#     # Show with matplotlib
#     display_img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#     plt.imshow(display_img)
#     plt.axis('off')
#     plt.title(f"Sample Index: {idx}")
#     plt.show()
    
#     3045 or 3046 is dark but outputted as not
#     bright scenes range from 88 - 115
#     Dark scenes range from ~20 - 80
#     dark scenes: ~2778

    
    
#     print("Is dark image: ", ("True" if img_is_dark else "False"))
#     print("Mean brightness: ", mean_brightness)
#     input("Press Enter to continue to the next sample...")
#     time.sleep(0.5)


Initializing NuScenes dataset with version v1.0-test at /storage/ice-shared/cs8803vlm/rmineyev3


In [35]:
output_dir = "/home/hice1/rmineyev3/scratch/VLM-Final-Project/eval_outputs"
header = "<40,<60,<80"
np.savetxt(output_dir + "/dark_scenes.csv", output_matrix, delimiter=",", fmt="%s", header=header, comments='')

percentage_under_40 = round(num_under_40/num_samples * 100, 2)
percentage_under_60 = round(num_under_60/num_samples * 100, 2)
percentage_under_80 = round(num_under_80/num_samples * 100, 2)

print("num under 40: ", num_under_40)
print("num under 60: ", num_under_60)
print("num under 80: ", num_under_80)
print("% under 40: ", percentage_under_40)
print("% under 60: ", percentage_under_60)
print("% under 80: ", percentage_under_80)

# For dark scenes:
# % under 20:  0.53
# % under 40:  36.58
# % under 60:  83.68
# % under 80:  99.74

num under 40:  139
num under 60:  318
num under 80:  379
% under 40:  4.4
% under 60:  10.07
% under 80:  12.0
